# UHC 가이드 관련 내용 전처리
- 각 파일마다 양식이 달라서 전처리 방식 다름
- 해당 파일 내 페이지마다 어떤 것은 글만 있고 표 형태로 있기도 해서 파일의 페이지별로 전처리도 다르게 함
    - 표 형태의 페이지는 별도로 pdfplumber를 통해 추출함

In [ ]:
#from pathlib import Path
#
#BASE_DIR = Path.cwd().parent
#
#GUIDE_DIR = BASE_DIR / "data" / "guide"
#OUTPUT_DIR = BASE_DIR / "outputs" / "guide"
#
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
#guide_files = list(GUIDE_DIR.glob("*.pdf"))
#
#print("guide PDF 개수:", len(guide_files))
#for f in guide_files:
#    print(f.name)

guide PDF 개수: 4
MBR-BT-1611172 Business Travel Member FAQs_221123_HRPrint.pdf
SAL-EU-1800500-12 BeHealthy SOB - Jan2024.pdf
UHC Global Program Guide.pdf
UHCG Expat Member Welcome Guide International 1222.pdf


In [ ]:
#from langchain_community.document_loaders import PyMuPDFLoader
#
#def load_pdf_pages(file_path):
#    loader = PyMuPDFLoader(str(file_path))
#    docs = loader.load()
#
#    pages = []
#
#    for doc in docs:
#        pages.append({
#            "page": doc.metadata.get("page", 0) + 1,
#            "text": doc.page_content
#        })
#
#    return pages

In [ ]:
#pages = load_pdf_pages(guide_files[0])
#
#print("페이지 수:", len(pages))
#print("첫 페이지 번호:", pages[0]["page"])
#print(pages[0]["text"][:1000])

페이지 수: 2
첫 페이지 번호: 1
Business Travel Insurance  |  FAQs
Frequently asked questions about your
business travel insurance plan
How do you access UnitedHealthcare Global business 
travel assistance services?
You can access the worldwide UnitedHealthcare Global 
network 24/7 by calling UnitedHealthcare Global using the 
toll-free or direct dial numbers printed on your business travel 
insurance fulfillment materials or your ID card.
When can I access UnitedHealthcare Global business 
travel assistance services?
UnitedHealthcare Global is available 24 hours a day, every
day of the year. We are here to help with any type of medical 
or travel inquiry, including lost documentation, regardless of 
the severity.
Where can I access UnitedHealthcare Global business 
travel assistance services?
UnitedHealthcare Global services extend worldwide. In the 
last 2 years, we have helped people in over 230 different 
countries and territories. However, in some countries the 
rendering of care or assistan

---

# UHCG Expat Member Welcome Guide International 파일 전처리

In [ ]:
#UHCG Expat Member Welcome Guide International 1222.pdf 
# 표 있는 페이지
#pages = load_pdf_pages("C://SKN_24//4차 프로젝트 개인//전처리//UHC//data//guide//UHCG Expat Member Welcome Guide International 1222.pdf")
#
#page_num =8
#text = pages[page_num - 1]["text"]
#
#print(text)
#print("Need help" in text)
#print("emergency" in text.lower())

8
If you need medical attention in select locations
Different countries have different rules and regulations when it comes to health care. An insurance claim has the potential to turn 
into a complicated maze of red tape due to language barriers, local laws, customs and norms that differ from country to country. 
We remove the complexity and partner with locally licensed insurers or administrators in countries where this type of coverage 
is required. All you need to do is show the right ID card to receive care, or contact us to arrange payment to a provider.
If you are living in or receiving care in one of the countries listed below, you may receive and need to carry an additional 
insurance ID card. Simply present the locally licensed insurer or administrator ID card at the time of service. Use your 
UnitedHealthcare Global ID card in all other instances.
To help you understand who your locally licensed insurer or administrator is, which ID card to use and who to call for 
assistance

In [10]:
target_file = None

for f in guide_files:
    if "Welcome Guide" in f.name:
        target_file = f

print(target_file)

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\data\guide\UHCG Expat Member Welcome Guide International 1222.pdf


In [11]:
pages = load_pdf_pages(target_file)

print("페이지 수:", len(pages))
print(pages[0]["text"][:500])

페이지 수: 16
Welcome
Explore the ways your international 
benefit plan can help you thrive
UnitedHealthcare Global Customer Support:
Client Name
+1.877.844.0280


In [12]:
import re
#줄바꿈 정리,공백제거, 특수문자 제거
def clean_text(text):
    text = text.replace("\x0c", " ")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"©.*", "", text)
    return text.strip()

In [13]:
#마케팅 등 문구 제거
def is_noise_text(text):
    text_lower = text.lower()

    noise_keywords = [
        "welcome",
        "explore the ways",
        "try a virtual visit",
        "learn more",
        "customer support",
        "client name",
    ]

    for kw in noise_keywords:
        if kw in text_lower:
            return True

    if len(text.strip()) < 30:
        return True

    return False

In [ ]:
# 표 있는 페이지 확인
import pdfplumber

def find_table_pages(file_path):
    table_pages = []

    with pdfplumber.open(file_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            if tables:
                table_pages.append(i)

    return table_pages

In [ ]:
#table_pages = find_table_pages(target_file)
#print(table_pages)

[1, 8, 12, 16]


In [16]:
EXCLUDE_PAGES = [1, 15, 16]
REAL_TABLE_PAGES = [8]

table_pages = REAL_TABLE_PAGES

print("제외 페이지:", EXCLUDE_PAGES)
print("실제 표 페이지:", table_pages)

제외 페이지: [1, 15, 16]
실제 표 페이지: [8]


In [17]:
INSURER = "UnitedHealthcare Global"

def make_welcome_text_chunks(file_path, table_pages):
    pages = load_pdf_pages(file_path)
    chunks = []

    for page in pages:
        page_num = page["page"]

        if page_num in EXCLUDE_PAGES:
            continue

        if page_num in table_pages:
            continue

        cleaned = clean_text(page["text"])

        if is_noise_text(cleaned):
            continue

        chunks.append({
            "chunk_id": f"{file_path.stem}_p{page_num}_text",
            "content": cleaned,
            "metadata": {
                "insurer": INSURER,
                "document_group": "guide",
                "document_type": "member_guide",
                "source_file": file_path.name,
                "page": page_num,
                "chunk_type": "page_text",
                "region_scope": "global",
                "korea_applicability": "conditional",
                "language": "en"
            }
        })

    return chunks

In [ ]:
#def extract_table_rows(file_path, page_num):
#    rows_data = []
#
#    with pdfplumber.open(file_path) as pdf:
#        page = pdf.pages[page_num - 1]
#        tables = page.extract_tables()
#
#        for table_idx, table in enumerate(tables, start=1):
#            if not table or len(table) < 2:
#                continue
#
#            header = table[0]
#            rows = table[1:]
#
#            for row_idx, row in enumerate(rows, start=1):
#                row_text_parts = []
#
#                for h, cell in zip(header, row):
#                    if h and cell:
#                        row_text_parts.append(f"{h.strip()}: {cell.strip()}")
#
#                if row_text_parts:
#                    rows_data.append({
#                        "table_idx": table_idx,
#                        "row_idx": row_idx,
#                        "content": "\n".join(row_text_parts)
#                    })
#
#    return rows_data

In [ ]:
def extract_table_rows(file_path, page_num):
    rows_data = []

    country_fallback = [
        "Africa",
        "Australia",
        "Bahrain, Jordan, Kuwait, Lebanon, Kingdom of Saudi Arabia, Oman, Qatar, United Arab Emirates",
        "Canada",
        "Europe, plus Austria, Belgium and Luxembourg",
        "Japan",
        "India"
    ]

    with pdfplumber.open(file_path) as pdf:
        page = pdf.pages[page_num - 1]
        tables = page.extract_tables()

        for table_idx, table in enumerate(tables, start=1):
            if not table or len(table) < 2:
                continue

            header = [
                "When you are in",
                "The locally licensed insurer or administrator will be",
                "Carry the following ID cards in this country",
                "For assistance, contact"
            ]

            rows = table[1:]

            for row_idx, row in enumerate(rows, start=1):
                row = list(row)

                while len(row) < 4:
                    row.append("")

                row = row[:4]

                # 첫 번째 열이 None이면 fallback 국가명 넣음
                if not row[0] and row_idx <= len(country_fallback):
                    row[0] = country_fallback[row_idx - 1]

                row_text_parts = []

                for h, cell in zip(header, row):
                    if cell and str(cell).strip():
                        row_text_parts.append(f"{h}: {str(cell).strip()}")

                if row_text_parts:
                    rows_data.append({
                        "table_idx": table_idx,
                        "row_idx": row_idx,
                        "content": "\n".join(row_text_parts)
                    })

    return rows_data

In [34]:
text_chunks = make_welcome_text_chunks(target_file, table_pages)
table_chunks = make_welcome_table_chunks(target_file, table_pages)

welcome_chunks = text_chunks + table_chunks

print("text chunk:", len(text_chunks))
print("table chunk:", len(table_chunks))
print("total:", len(welcome_chunks))

for c in welcome_chunks:
    print(c["metadata"]["page"], c["metadata"]["chunk_type"], c["chunk_id"])

text chunk: 6
table chunk: 7
total: 13
4 page_text UHCG Expat Member Welcome Guide International 1222_p4_text
6 page_text UHCG Expat Member Welcome Guide International 1222_p6_text
7 page_text UHCG Expat Member Welcome Guide International 1222_p7_text
9 page_text UHCG Expat Member Welcome Guide International 1222_p9_text
10 page_text UHCG Expat Member Welcome Guide International 1222_p10_text
12 page_text UHCG Expat Member Welcome Guide International 1222_p12_text
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row1
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row2
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row3
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row4
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row5
8 table_row UHCG Expat Member Welcome Guide International 1222_p8_table1_row6
8 table_row UHCG Expat Member Welcome Guide International 1222_

In [35]:
#확인용
for c in welcome_chunks:
    if c["metadata"]["chunk_type"] == "table_row":
        print(c["content"])
        break

When you are in: Africa
The locally licensed insurer or administrator will be: Medical Services
Organization (MSO)
Carry the following ID cards in this country: UnitedHealthcare Global


In [36]:
#확인용
for c in welcome_chunks:
    if c["metadata"]["chunk_type"] == "table_row":
        print("===")
        print(c["chunk_id"])
        print(c["content"])

===
UHCG Expat Member Welcome Guide International 1222_p8_table1_row1
When you are in: Africa
The locally licensed insurer or administrator will be: Medical Services
Organization (MSO)
Carry the following ID cards in this country: UnitedHealthcare Global
===
UHCG Expat Member Welcome Guide International 1222_p8_table1_row2
When you are in: Australia
The locally licensed insurer or administrator will be: nib Health Funds (nib)
Carry the following ID cards in this country: • UnitedHealthcare Global
• nib
===
UHCG Expat Member Welcome Guide International 1222_p8_table1_row3
When you are in: Bahrain, Jordan, Kuwait, Lebanon, Kingdom of Saudi Arabia, Oman, Qatar, United Arab Emirates
The locally licensed insurer or administrator will be: Al Sagr National Insurance
Company (ASNIC) with
NEXtCARE, a local third-
party administrator who will
process your claims and
provide customer support
Carry the following ID cards in this country: • UnitedHealthcare Global
• ASNIC
===
UHCG Expat Member Welc

In [ ]:
#원본 확인용
#rows = extract_table_rows(target_file, 8)
#
#with pdfplumber.open(target_file) as pdf:
#    page = pdf.pages[7]
#    tables = page.extract_tables()
#    for i, row in enumerate(tables[0][:5]):
#        print(i, len(row), row)

0 4 ['When you are in:', 'The locally licensed insurer\nor administrator will be:', 'Carry the following ID cards\nin this country:', 'For assistance, contact:']
1 4 [None, 'Medical Services\nOrganization (MSO)', 'UnitedHealthcare Global', None]
2 4 [None, 'nib Health Funds (nib)', '• UnitedHealthcare Global\n• nib', None]
3 4 [None, 'Al Sagr National Insurance\nCompany (ASNIC) with\nNEXtCARE, a local third-\nparty administrator who will\nprocess your claims and\nprovide customer support', '• UnitedHealthcare Global\n• ASNIC', None]
4 4 [None, 'Cowan', '• UnitedHealthcare Global\n• Cowan', None]


In [37]:
import json

output_path = OUTPUT_DIR / "welcome_guide_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(welcome_chunks, f, ensure_ascii=False, indent=2)

print(output_path)
print(output_path.exists())

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\outputs\guide\welcome_guide_chunks.json
True


---

# SAL-EU-1800500-12 BeHealthy SOB - Jan2024 파일 전처리
- 표로만 이루어진 파일임
- 각 프랜별 보장 한도와 보장 여부 비교

In [38]:
target_file = None

for f in guide_files:
    if "BeHealthy SOB" in f.name or "SAL-EU" in f.name:
        target_file = f

print(target_file)

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\data\guide\SAL-EU-1800500-12 BeHealthy SOB - Jan2024.pdf


In [39]:
import pdfplumber

In [40]:
with pdfplumber.open(target_file) as pdf:
    print("페이지 수:", len(pdf.pages))

    for page_num, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()
        print(page_num, "페이지 표 개수:", len(tables))

페이지 수: 7
1 페이지 표 개수: 1
2 페이지 표 개수: 3
3 페이지 표 개수: 2
4 페이지 표 개수: 2
5 페이지 표 개수: 1
6 페이지 표 개수: 2
7 페이지 표 개수: 0


In [48]:
with pdfplumber.open(target_file) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

    print("1페이지 표 개수:", len(tables))

    for row in tables[0][:10]:
        print(row)



1페이지 표 개수: 1
['Core Plan', 'BeHealthy Core 1', 'BeHealthy Core 2', 'BeHealthy Core 3']
['Annual Maximum Benefit USD ($)', '$1,500,000', '$3,000,000', 'No Limit']
['Annual Maximum Benefit EUR (€)', '€1,300,000', '€2,600,000', 'No Limit']
['Annual Maximum Benefit GBP (£)', '£1,200,000', '£2,400,000', 'No Limit']
['Annual Maximum Benefit CHF', '1,400,000 CHF', '2,800,000 CHF', 'No Limit']
[None, '', '', None]
['Core Plan Healthcare Benefits', 'BeHealthy Core 1', 'BeHealthy Core 2', 'BeHealthy Core 3']
['Hospital Accommodation*', 'Private Room', 'Private Room', 'Private Room']
['Day-patient Treatment*', 'Paid in Full', 'Paid in Full', 'Paid in Full']
['Prescriptions Medicines, Drugs and Dressings*', 'Paid in Full', 'Paid in Full', 'Paid in Full']


In [ ]:
# 표 행 추출
def extract_sob_table_rows(file_path):
    rows_data = []

    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()

            for table_idx, table in enumerate(tables, start=1):
                if not table or len(table) < 2:
                    continue

                header = table[0]
                rows = table[1:]

                for row_idx, row in enumerate(rows, start=1):
                    row = list(row)

                    while len(row) < len(header):
                        row.append("")

                    row = row[:len(header)]

                    row_text_parts = []

                    for h, cell in zip(header, row):
                        if h and cell and str(cell).strip():
                            row_text_parts.append(
                                f"{str(h).strip()}: {str(cell).strip()}"
                            )

                    if row_text_parts:
                        rows_data.append({
                            "page": page_num,
                            "table_idx": table_idx,
                            "row_idx": row_idx,
                            "content": "\n".join(row_text_parts)
                        })

    return rows_data

In [50]:
sob_rows = extract_sob_table_rows(target_file)

print("추출 row 개수:", len(sob_rows))

for r in sob_rows[:10]:
    print("===")
    print("page:", r["page"], "table:", r["table_idx"], "row:", r["row_idx"])
    print(r["content"])

추출 row 개수: 83
===
page: 1 table: 1 row: 1
Core Plan: Annual Maximum Benefit USD ($)
BeHealthy Core 1: $1,500,000
BeHealthy Core 2: $3,000,000
BeHealthy Core 3: No Limit
===
page: 1 table: 1 row: 2
Core Plan: Annual Maximum Benefit EUR (€)
BeHealthy Core 1: €1,300,000
BeHealthy Core 2: €2,600,000
BeHealthy Core 3: No Limit
===
page: 1 table: 1 row: 3
Core Plan: Annual Maximum Benefit GBP (£)
BeHealthy Core 1: £1,200,000
BeHealthy Core 2: £2,400,000
BeHealthy Core 3: No Limit
===
page: 1 table: 1 row: 4
Core Plan: Annual Maximum Benefit CHF
BeHealthy Core 1: 1,400,000 CHF
BeHealthy Core 2: 2,800,000 CHF
BeHealthy Core 3: No Limit
===
page: 1 table: 1 row: 6
Core Plan: Core Plan Healthcare Benefits
BeHealthy Core 1: BeHealthy Core 1
BeHealthy Core 2: BeHealthy Core 2
BeHealthy Core 3: BeHealthy Core 3
===
page: 1 table: 1 row: 7
Core Plan: Hospital Accommodation*
BeHealthy Core 1: Private Room
BeHealthy Core 2: Private Room
BeHealthy Core 3: Private Room
===
page: 1 table: 1 row: 8
Core P

In [52]:
#metadata 포함 chunk 생성
def make_behealthy_sob_chunks(file_path):
    rows = extract_sob_table_rows(file_path)
    chunks = []

    for r in rows:
        chunks.append({
            "chunk_id": f"{file_path.stem}_p{r['page']}_table{r['table_idx']}_row{r['row_idx']}",
            "content": r["content"],
            "metadata": {
                "insurer": INSURER,
                "document_group": "guide",
                "document_type": "benefit_summary",
                "source_file": file_path.name,
                "page": r["page"],
                "chunk_type": "benefit_table_row",
                "region_scope": "global",
                "korea_applicability": "conditional",
                "language": "en"
            }
        })

    return chunks

In [53]:
sob_chunks = make_behealthy_sob_chunks(target_file)

print("SOB chunk 개수:", len(sob_chunks))

for c in sob_chunks[:10]:
    print("===")
    print(c["chunk_id"])
    print(c["metadata"])
    print(c["content"])

SOB chunk 개수: 83
===
SAL-EU-1800500-12 BeHealthy SOB - Jan2024_p1_table1_row1
{'insurer': 'UnitedHealthcare Global', 'document_group': 'guide', 'document_type': 'benefit_summary', 'source_file': 'SAL-EU-1800500-12 BeHealthy SOB - Jan2024.pdf', 'page': 1, 'chunk_type': 'benefit_table_row', 'region_scope': 'global', 'korea_applicability': 'conditional', 'language': 'en'}
Core Plan: Annual Maximum Benefit USD ($)
BeHealthy Core 1: $1,500,000
BeHealthy Core 2: $3,000,000
BeHealthy Core 3: No Limit
===
SAL-EU-1800500-12 BeHealthy SOB - Jan2024_p1_table1_row2
{'insurer': 'UnitedHealthcare Global', 'document_group': 'guide', 'document_type': 'benefit_summary', 'source_file': 'SAL-EU-1800500-12 BeHealthy SOB - Jan2024.pdf', 'page': 1, 'chunk_type': 'benefit_table_row', 'region_scope': 'global', 'korea_applicability': 'conditional', 'language': 'en'}
Core Plan: Annual Maximum Benefit EUR (€)
BeHealthy Core 1: €1,300,000
BeHealthy Core 2: €2,600,000
BeHealthy Core 3: No Limit
===
SAL-EU-1800500-

In [54]:
import json

output_path = OUTPUT_DIR / "behealthy_sob_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(sob_chunks, f, ensure_ascii=False, indent=2)

print(output_path)
print(output_path.exists())

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\outputs\guide\behealthy_sob_chunks.json
True


In [55]:
with open(output_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("저장된 chunk 개수:", len(loaded))
print(loaded[0].keys())
print(loaded[0]["metadata"])
print(loaded[0]["content"])

저장된 chunk 개수: 83
dict_keys(['chunk_id', 'content', 'metadata'])
{'insurer': 'UnitedHealthcare Global', 'document_group': 'guide', 'document_type': 'benefit_summary', 'source_file': 'SAL-EU-1800500-12 BeHealthy SOB - Jan2024.pdf', 'page': 1, 'chunk_type': 'benefit_table_row', 'region_scope': 'global', 'korea_applicability': 'conditional', 'language': 'en'}
Core Plan: Annual Maximum Benefit USD ($)
BeHealthy Core 1: $1,500,000
BeHealthy Core 2: $3,000,000
BeHealthy Core 3: No Limit


---

# UHC Global Program Guide 파일 전처리
- 전반적으로 텍스트

In [56]:
target_file = None

for f in guide_files:
    if "Program Guide" in f.name or "UHC Global Program Guide" in f.name:
        target_file = f

print(target_file)

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\data\guide\UHC Global Program Guide.pdf


In [57]:
pages = load_pdf_pages(target_file)

print("페이지 수:", len(pages))
print(pages[0]["text"][:1000])

페이지 수: 9
UHCG-MSSNDS-UHCSR-1214 
GLOBAL EMERGENCY SERVICES 
 
This program provides the Participant (and spouse and/or dependent children if enrolled) with access to doctors, 
hospitals, pharmacies and certain other services when faced with a travel or medical emergency while traveling 100 
miles or more from his/her permanent residence or abroad.  The program also provides emergency security and 
natural disaster assistance services when you are outside of your home country.  For international students, all 
medical, security, and natural disaster services are available only when outside of your home country. 
 
One phone call to UnitedHealthcare Global connects the student to: 
 
A state-of-the-art Emergency Response Center with worldwide response capabilities 
 
Experienced crisis management professionals 
 
A global network of over 41,000 pre-qualified medical providers 
 
Air and ground ambulance service providers 
 
UnitedHealthcare Global arranges and pays for all Medical Ev

In [72]:
def clean_program_text(text):
    text = clean_text(text)

    # 문서 코드/깨진 문자 제거
    text = text.replace("UHCG-MSSNDS-UHCSR-1214", "")
    text = text.replace("￾", "")
    text = text.replace("�", "-")
    text = text.replace("", "-")

    # bullet 비슷한 깨짐 정리
    bullet_chars = ["", "", "▪", "●", "•", "◦", "·"]
    for b in bullet_chars:
        text = text.replace(b, "-")


    # 공백/줄바꿈 재정리
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"-\s*\n\s*", "- ", text)

    return text.strip()

In [75]:
cleaned = clean_program_text(pages[0]["text"])

print(cleaned[:1200])
print("bullet:", any(b in cleaned for b in ["", "", "▪", "●", "•", "◦", "·"]))

GLOBAL EMERGENCY SERVICES 
 
This program provides the Participant (and spouse and/or dependent children if enrolled) with access to doctors, 
hospitals, pharmacies and certain other services when faced with a travel or medical emergency while traveling 100 
miles or more from his/her permanent residence or abroad. The program also provides emergency security and 
natural disaster assistance services when you are outside of your home country. For international students, all 
medical, security, and natural disaster services are available only when outside of your home country. 
 
One phone call to UnitedHealthcare Global connects the student to: 
- A state-of-the-art Emergency Response Center with worldwide response capabilities 
- Experienced crisis management professionals 
- A global network of over 41,000 pre-qualified medical providers 
- Air and ground ambulance service providers 
 
UnitedHealthcare Global arranges and pays for all Medical Evacuation and Repatriation Services it p

In [ ]:
import re
#섹션별 제목 리스트
SECTION_TITLES = [
    "GLOBAL EMERGENCY SERVICES",
    "PROGRAM GUIDELINES",
    "MEDICAL & SECURITY ASSISTANCE AND EVACUATION",
    "PROGRAM DESCRIPTION",
    "How To Use UnitedHealthcare Global Assistance Services",
    "MEDICAL ASSISTANCE SERVICES",
    "MEDICAL EVACUATION & REPATRIATION SERVICES",
    "WORLDWIDE DESTINATION INTELLIGENCE",
    "SECURITY AND POLITICAL EVACUATION SERVICES",
    "NATURAL DISASTER EVACUATION SERVICES",
    "TRAVEL ASSISTANCE SERVICES",
    "PROGRAM DEFINITIONS",
    "CONDITIONS AND LIMITATIONS",
]

In [77]:

#section 분리 함수 추가
def split_program_sections(text):
    pattern = "|".join([re.escape(title) for title in SECTION_TITLES])
    parts = re.split(f"({pattern})", text)

    sections = []

    for i in range(1, len(parts), 2):
        title = parts[i].strip()
        content = parts[i + 1].strip() if i + 1 < len(parts) else ""

        if len(content) >= 30:
            sections.append({
                "section_title": title,
                "content": content
            })

    return sections

In [78]:
#확인용
full_text = "\n".join([clean_program_text(p["text"]) for p in pages])
sections = split_program_sections(full_text)

print("section 개수:", len(sections))

for s in sections:
    print("===")
    print(s["section_title"])
    print(s["content"][:300])

section 개수: 13
===
GLOBAL EMERGENCY SERVICES
This program provides the Participant (and spouse and/or dependent children if enrolled) with access to doctors, 
hospitals, pharmacies and certain other services when faced with a travel or medical emergency while traveling 100 
miles or more from his/her permanent residence or abroad. The program 
===
PROGRAM GUIDELINES
Students studying outside the U.S. – you are eligible for services both at and away from your campus location 
during your 2015-2016 UnitedHealthcare StudentResources policy period, however, you must be at least 100 miles 
away from your permanent residence. 
U.S. students studying in U.S. location 
===
MEDICAL & SECURITY ASSISTANCE AND EVACUATION
MEDICAL, SECURITY, AND NATURAL DISASTER SERVICE
===
PROGRAM DESCRIPTION
A comprehensive program providing 24/7 emergency medical and travel assistance services when You are outside 
Your Home Country or 100 or more miles away from Your primary residence or campus in Your Home Coun

In [79]:
def make_program_guide_chunks(file_path):
    pages = load_pdf_pages(file_path)
    full_text = "\n".join([clean_program_text(p["text"]) for p in pages])
    sections = split_program_sections(full_text)

    chunks = []

    for idx, section in enumerate(sections, start=1):
        section_title = section["section_title"]
        content = section["content"]

        chunks.append({
            "chunk_id": f"{file_path.stem}_section_{idx}",
            "content": f"{section_title}\n{content}",
            "metadata": {
                "insurer": INSURER,
                "document_group": "guide",
                "document_type": "program_guide",
                "source_file": file_path.name,
                "page": None,
                "chunk_type": "section",
                "section_title": section_title,
                "region_scope": "global",
                "korea_applicability": "conditional",
                "language": "en"
            }
        })

    return chunks

In [80]:
program_chunks = make_program_guide_chunks(target_file)

print("Program Guide chunk 개수:", len(program_chunks))

for c in program_chunks:
    print("===")
    print(c["chunk_id"])
    print(c["metadata"]["section_title"])
    print(c["content"][:300])

Program Guide chunk 개수: 13
===
UHC Global Program Guide_section_1
GLOBAL EMERGENCY SERVICES
GLOBAL EMERGENCY SERVICES
This program provides the Participant (and spouse and/or dependent children if enrolled) with access to doctors, 
hospitals, pharmacies and certain other services when faced with a travel or medical emergency while traveling 100 
miles or more from his/her permanent residen
===
UHC Global Program Guide_section_2
PROGRAM GUIDELINES
PROGRAM GUIDELINES
Students studying outside the U.S. – you are eligible for services both at and away from your campus location 
during your 2015-2016 UnitedHealthcare StudentResources policy period, however, you must be at least 100 miles 
away from your permanent residence. 
U.S. students studyin
===
UHC Global Program Guide_section_3
MEDICAL & SECURITY ASSISTANCE AND EVACUATION
MEDICAL & SECURITY ASSISTANCE AND EVACUATION
MEDICAL, SECURITY, AND NATURAL DISASTER SERVICE
===
UHC Global Program Guide_section_4
PROGRAM DESCRIPTION
PROGRAM DESC

In [81]:
import json

output_path = OUTPUT_DIR / "program_guide_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(program_chunks, f, ensure_ascii=False, indent=2)

print(output_path)
print(output_path.exists())

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\outputs\guide\program_guide_chunks.json
True


In [82]:
with open(output_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("저장된 chunk 개수:", len(loaded))
print(loaded[0]["metadata"])
print(loaded[0]["content"][:500])

저장된 chunk 개수: 13
{'insurer': 'UnitedHealthcare Global', 'document_group': 'guide', 'document_type': 'program_guide', 'source_file': 'UHC Global Program Guide.pdf', 'page': None, 'chunk_type': 'section', 'section_title': 'GLOBAL EMERGENCY SERVICES', 'region_scope': 'global', 'korea_applicability': 'conditional', 'language': 'en'}
GLOBAL EMERGENCY SERVICES
This program provides the Participant (and spouse and/or dependent children if enrolled) with access to doctors, 
hospitals, pharmacies and certain other services when faced with a travel or medical emergency while traveling 100 
miles or more from his/her permanent residence or abroad. The program also provides emergency security and 
natural disaster assistance services when you are outside of your home country. For international students, all 
medical, security, and 


In [83]:
import json

output_path = OUTPUT_DIR / "program_guide_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(program_chunks, f, ensure_ascii=False, indent=2)

print(output_path)
print(output_path.exists())

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\outputs\guide\program_guide_chunks.json
True


---

# MBR-BT-1611172 Business Travel Member FAQs_221123_HRPrint 파일 전처리

In [87]:
target_file=None
for f in guide_files:
    if "Business Travel Mebmber FAQs" in f.name or "MBR-BT" in f.name:
        target_file =f
print(target_file)


c:\SKN_24\4차 프로젝트 개인\전처리\UHC\data\guide\MBR-BT-1611172 Business Travel Member FAQs_221123_HRPrint.pdf


In [90]:
pages = load_pdf_pages(target_file)

print("페이지 수:", len(pages))
print(pages[1]["text"][:100000])

페이지 수: 2
We will consult with all parties involved and fully manage the 
issues surrounding the evacuation.
What happens when I am released from hospital?
UnitedHealthcare Global assists with your case until
the point when you have returned home or have received
final treatment.
What is involved in an evacuation?
The choice of transportation, from a commercial airline to a 
dedicated air ambulance, will be dictated by your condition 
and location.
Our medical management team will coordinate all aspects of 
the process to support a positive medical outcome including:
•	Evaluation of the transport requirements (such as oxygen 
requirements, doctors necessary, any special equipment, 
altitude specifications, etc.)
•	Discharge administration
•	Admission into a new facility
•	Identifying qualified aeromedical escorts and
air ambulances
•	Coordinating ground transportation on both ends of
the evacuation
•	Immigration and flight clearances
•	Assistance with travel arrangements if required, fo

In [91]:
def clean_faq_text(text):
    text = clean_text(text)

    text = text.replace("Business Travel Insurance | FAQs", "")
    text = text.replace("Frequently asked questions about your business travel insurance plan", "")
    text = text.replace("continued", "")

    text = re.sub(r"©.*", "", text)
    text = re.sub(r"\d{2}/\d{2}\s+MBR-BT-\d+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [92]:
cleaned = clean_faq_text(pages[0]["text"])
print(cleaned[:1000])

Business Travel Insurance  |  FAQs
Frequently asked questions about your
business travel insurance plan
How do you access UnitedHealthcare Global business 
travel assistance services?
You can access the worldwide UnitedHealthcare Global 
network 24/7 by calling UnitedHealthcare Global using the 
toll-free or direct dial numbers printed on your business travel 
insurance fulfillment materials or your ID card.
When can I access UnitedHealthcare Global business 
travel assistance services?
UnitedHealthcare Global is available 24 hours a day, every
day of the year. We are here to help with any type of medical 
or travel inquiry, including lost documentation, regardless of 
the severity.
Where can I access UnitedHealthcare Global business 
travel assistance services?
UnitedHealthcare Global services extend worldwide. In the 
last 2 years, we have helped people in over 230 different 
countries and territories. However, in some countries the 
rendering of care or assistance services is prohib

In [93]:
#Q/A 분리 함수
def split_faq_qa(text):
    question_pattern = r"(?=^[A-Z][^\n?]+\?)"
    blocks = re.split(question_pattern, text, flags=re.MULTILINE)

    qa_list = []

    for block in blocks:
        block = block.strip()

        if "?" not in block:
            continue

        question, answer = block.split("?", 1)
        question = question.strip() + "?"
        answer = answer.strip()

        if len(question) < 10 or len(answer) < 20:
            continue

        qa_list.append({
            "question": question,
            "answer": answer
        })

    return qa_list

In [94]:
#Q/A 분리 확인
full_text = "\n".join([clean_faq_text(p["text"]) for p in pages])
qa_list = split_faq_qa(full_text)

print("FAQ 개수:", len(qa_list))

for qa in qa_list[:5]:
    print("===")
    print("Q:", qa["question"])
    print("A:", qa["answer"][:300])

FAQ 개수: 7
===
Q: Business Travel Insurance  |  FAQs
Frequently asked questions about your
business travel insurance plan
How do you access UnitedHealthcare Global business 
travel assistance services?
A: You can access the worldwide UnitedHealthcare Global 
network 24/7 by calling UnitedHealthcare Global using the 
toll-free or direct dial numbers printed on your business travel 
insurance fulfillment materials or your ID card.
When can I access UnitedHealthcare Global business 
travel assistance se
===
Q: What happens in the event of a hospitalization?
A: Please notify us as soon as possible in the event of 
hospitalization. We will speak immediately with your treating 
doctor to assess your condition, your treatment plans, and 
whether or not an evacuation is necessary. We will update 
your family, employer and personal physician as appropriate 
and
===
Q: What happens when I am released from hospital?
A: UnitedHealthcare Global assists with your case until
the point when you have re

In [95]:
def business_travel_faq_chunks(file_path):
    pages = load_pdf_pages(file_path)
    full_text = "\n".join([clean_faq_text(p["text"]) for p in pages])
    qa_list = split_faq_qa(full_text)

    chunks = []

    for idx, qa in enumerate(qa_list, start=1):
        chunks.append({
            "chunk_id": f"{file_path.stem}_faq_{idx}",
            "content": f"Question: {qa['question']}\nAnswer: {qa['answer']}",
            "metadata": {
                "insurer": INSURER,
                "document_group": "guide",
                "document_type": "faq",
                "source_file": file_path.name,
                "page": None,
                "chunk_type": "faq_qa",
                "question": qa["question"],
                "region_scope": "global",
                "korea_applicability": "conditional",
                "language": "en"
            }
        })

    return chunks

In [96]:
faq_chunks = business_travel_faq_chunks(target_file)

print("FAQ chunk 개수:", len(faq_chunks))

for c in faq_chunks[:5]:
    print("===")
    print(c["chunk_id"])
    print(c["metadata"]["question"])
    print(c["content"][:300])

FAQ chunk 개수: 7
===
MBR-BT-1611172 Business Travel Member FAQs_221123_HRPrint_faq_1
Business Travel Insurance  |  FAQs
Frequently asked questions about your
business travel insurance plan
How do you access UnitedHealthcare Global business 
travel assistance services?
Question: Business Travel Insurance  |  FAQs
Frequently asked questions about your
business travel insurance plan
How do you access UnitedHealthcare Global business 
travel assistance services?
Answer: You can access the worldwide UnitedHealthcare Global 
network 24/7 by calling UnitedHealthcare Glo
===
MBR-BT-1611172 Business Travel Member FAQs_221123_HRPrint_faq_2
What happens in the event of a hospitalization?
Question: What happens in the event of a hospitalization?
Answer: Please notify us as soon as possible in the event of 
hospitalization. We will speak immediately with your treating 
doctor to assess your condition, your treatment plans, and 
whether or not an evacuation is necessary. We will update
===
MBR-BT-161

In [97]:
import json

output_path = OUTPUT_DIR / "business_travel_faq_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(faq_chunks, f, ensure_ascii=False, indent=2)

print(output_path)
print(output_path.exists())

c:\SKN_24\4차 프로젝트 개인\전처리\UHC\outputs\guide\business_travel_faq_chunks.json
True
